# SmolVLA + Gated Attention — backbone swap: SmolVLM2 → LFM2.5-VL-450M (8 layers)

Unified notebook. What changes vs. `smol-but-gate-vla`: **only the vision-language backbone**.

| | |
|---|---|
| Backbone | `LiquidAI/LFM2.5-VL-450M`, first **8** LM layers, frozen (replaces SmolVLM2-500M) |
| Adapter | `AdaptiveConditionalGatedKV` from `lfm2-backbone-new-dataset` (frozen features → per-layer prefix K/V for the action expert) |
| Unchanged | gated attention (G1, elementwise, bias-init 4.0) in all 16 expert layers, action expert, state/action/time projections, flow-matching loss, dataset, optimizer, LR schedule, batch size, steps, seed, `lerobot-train` pipeline |

Cells carried over unchanged from the original notebook keep their original recorded outputs. Every cell marked **NEW** has no output yet: it was authored without GPU access, so run the notebook top to bottom on Kaggle (2×T4, internet on) to populate them.

In [1]:
import os
CODE_DIR = "/kaggle/working"
DATASET  = "MinhDuk/Chunking_571_Skynet_combined"
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_LEROBOT_HOME"] = "/tmp/lerobot"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
# NEW — backbone configuration (read by lfm_backbone_adapter.py in every process)
import os
os.environ["LFM_BACKBONE_ID"] = "LiquidAI/LFM2.5-VL-450M"
os.environ["LFM_KEEP_LAYERS"] = "8"
os.environ["LFM_VIS_TOKENS_PER_CAM"] = "48"   # same visual-token budget as the LFM2 reference notebook

In [2]:
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"
# lerobot needs av.option -> av 19 lacks it, pin av 15.x AFTER lerobot is installed
!pip install -q -U "av>=15,<16"
!python -c "import av, av.option; print('av', av.__version__, 'OK')"
!nvidia-smi -L

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
av 15.1.0 OK
GPU 0: Tesla T4 (UUID: GPU-9524f258-f2cd-e199-db50-36974898f0b8)
GPU 1: Tesla T4 (UUID: GPU-187c0b28-1207-55c6-7c9f-49a2307a684b)


In [ ]:
# NEW — LFM2.5-VL needs transformers >= 5.1 (checked in a subprocess: no kernel restart needed)
import subprocess, sys
get_v = lambda: subprocess.run([sys.executable, "-c", "import transformers; print(transformers.__version__)"],
                               capture_output=True, text=True).stdout.strip()
ver = get_v()
print("transformers", ver)
if tuple(int(x) for x in ver.split(".")[:2]) < (5, 1):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=5.5.0,<6"], check=True)
    print("upgraded ->", get_v())
!python -c "from transformers import Lfm2VlForConditionalGeneration, AutoProcessor; print('Lfm2VlForConditionalGeneration import OK')"

In [3]:
%%writefile /kaggle/working/gate_patch.py
#!/usr/bin/env python
"""
Patch LeRobot's SmolVLA in place: Gated Attention (Qiu et al., 2025, Qwen) in the ACTION EXPERT.

  Y' = Y * sigmoid(X @ W + b)      (paper position "G1": after SDPA, before o_proj)

  * X = pre-normed hidden state of the query token (expert/action token) -> query-dependent gate
  * head-specific, elementwise by default (one score per head-dim channel; best variant in the paper)
  * applied in EVERY expert layer (self-attention layers and cross-attention layers)
  * VLM layers / prefix tokens are never gated
  * gate init: W ~ N(0, 0.02), b = +4  ->  sigmoid ~ 0.98 (near identity, so the pretrained
    smolvla_base expert is not destroyed at step 0, but gradients still flow)
  * SmolVLM + expert are run in fp32 (T4 has no native bf16)

Nothing else is changed (masks, RoPE, KV cache, flow-matching loss, optimizer, scheduler).

    python gate_patch.py            # patch installed lerobot
    python gate_patch.py --dir D    # patch a given smolvla folder
    python gate_patch.py --revert   # restore .orig backups
"""
import argparse
import importlib.util
import pathlib
import shutil
import sys


def find_dir() -> pathlib.Path:
    spec = importlib.util.find_spec("lerobot")
    if spec is None or not spec.submodule_search_locations:
        sys.exit("lerobot is not installed.")
    return pathlib.Path(list(spec.submodule_search_locations)[0]) / "policies" / "smolvla"


def sub(text: str, old: str, new: str, name: str) -> str:
    n = text.count(old)
    if n != 1:
        sys.exit(f"[patch failed] anchor '{name}' found {n}x (expected 1). "
                 f"Your lerobot version differs from the one this patch was written for.")
    return text.replace(old, new)


# ------------------------------------------------------------------ configuration_smolvla.py
def patch_config(t: str) -> str:
    return sub(
        t,
        '    compile_mode: str = "max-autotune"  # Torch compile mode\n',
        '    compile_mode: str = "max-autotune"  # Torch compile mode\n'
        "\n"
        "    # Gated Attention (sigmoid gate after SDPA) in every layer of the action expert\n"
        "    use_gated_attn: bool = False\n"
        "    gated_attn_headwise: bool = False  # False: elementwise (n x heads x head_dim), True: one scalar per head\n"
        "    gated_attn_bias_init: float = 4.0  # sigmoid(4) = 0.982 -> near-identity start from the pretrained expert\n",
        "config.fields",
    )


# ------------------------------------------------------------------ smolvlm_with_expert.py
GATE_CLASS = '''class AttnOutputGate(nn.Module):
    """Gated Attention, position G1 (Qiu et al. 2025): Y' = Y * sigmoid(X W + b).
    Y = SDPA output [B, L, H*D] (before o_proj), X = normed hidden state of the query tokens [B, L, E].
    Head-specific. Elementwise: one score per channel (H*D). Headwise: one score per head (H)."""

    def __init__(self, hidden_size, num_heads, head_dim, headwise=False, bias_init=4.0):
        super().__init__()
        self.headwise = headwise
        self.head_dim = head_dim
        self.proj = nn.Linear(hidden_size, num_heads if headwise else num_heads * head_dim)
        nn.init.normal_(self.proj.weight, std=0.02)
        nn.init.constant_(self.proj.bias, bias_init)

    def forward(self, x, att_out):
        g = torch.sigmoid(self.proj(x.to(self.proj.weight.dtype)))
        if self.headwise:
            g = g.repeat_interleave(self.head_dim, dim=-1)
        return att_out * g.to(att_out.dtype)


class SmolVLMWithExpertModel(nn.Module):'''

GATE_BUILD = '''        # Remove unused embed_tokens
        self.lm_expert.embed_tokens = None

        # Gated attention: one gate per action-expert layer (self- and cross-attention layers)
        self.use_gated_attn = use_gated_attn
        self.attn_gates = nn.ModuleDict()
        if use_gated_attn:
            assert self.num_expert_layers == self.num_vlm_layers, (
                "Gated attention assumes num_expert_layers == num_vlm_layers (default config)"
            )
            for _i in range(self.num_expert_layers):
                self.attn_gates[str(_i)] = AttnOutputGate(
                    lm_expert_config.hidden_size,
                    lm_expert_config.num_attention_heads,
                    lm_expert_config.head_dim,
                    gated_attn_headwise,
                    gated_attn_bias_init,
                )
'''


def patch_vlm(t: str) -> str:
    t = sub(t, "class SmolVLMWithExpertModel(nn.Module):", GATE_CLASS, "class")
    t = sub(t, 'torch_dtype="bfloat16",', 'torch_dtype="float32",', "fp32 loader")
    t = sub(
        t,
        '        device: str = "auto",\n    ):\n        super().__init__()\n        require_package("transformers", extra="smolvla")',
        '        device: str = "auto",\n'
        "        use_gated_attn: bool = False,\n"
        "        gated_attn_headwise: bool = False,\n"
        "        gated_attn_bias_init: float = 4.0,\n"
        '    ):\n        super().__init__()\n        require_package("transformers", extra="smolvla")',
        "init signature",
    )
    t = sub(
        t,
        "        # Remove unused embed_tokens\n        self.lm_expert.embed_tokens = None\n",
        GATE_BUILD,
        "gate build",
    )
    # --- self-attention layers (joint prefix+suffix attention): remember the normed expert hidden state
    t = sub(
        t,
        "        query_states = []\n        key_states = []\n        value_states = []\n",
        "        gate_x = None\n        query_states = []\n        key_states = []\n        value_states = []\n",
        "gate_x init",
    )
    t = sub(
        t,
        "            hidden_states = layer.input_layernorm(hidden_states)\n",
        "            hidden_states = layer.input_layernorm(hidden_states)\n"
        "            if i == 1 and self.use_gated_attn:  # expert (action) tokens only\n"
        "                gate_x = hidden_states\n",
        "gate_x capture",
    )
    t = sub(
        t,
        "        att_output = attention_interface(\n"
        "            attention_mask_, batch_size, head_dim, query_states, key_states, value_states\n"
        "        )\n"
        "        return [att_output], past_key_values\n",
        "        att_output = attention_interface(\n"
        "            attention_mask_, batch_size, head_dim, query_states, key_states, value_states\n"
        "        )\n"
        "        if gate_x is not None:  # gate only the expert tokens (the last Ls query positions)\n"
        "            _ls = gate_x.shape[1]\n"
        "            _gated = self.attn_gates[str(layer_idx)](gate_x, att_output[:, -_ls:])\n"
        "            att_output = torch.cat([att_output[:, :-_ls], _gated], dim=1)\n"
        "        return [att_output], past_key_values\n",
        "self-attn gate apply",
    )
    # --- cross-attention layers (expert queries attend to the VLM KV)
    t = sub(
        t,
        "                expert_value_states,\n            )\n            att_outputs.append(att_output)\n",
        "                expert_value_states,\n            )\n"
        "            if self.use_gated_attn:\n"
        "                att_output = self.attn_gates[str(layer_idx)](expert_hidden_states, att_output)\n"
        "            att_outputs.append(att_output)\n",
        "cross-attn gate apply",
    )
    return t


# ------------------------------------------------------------------ modeling_smolvla.py
def patch_modeling(t: str) -> str:
    t = sub(
        t,
        '            device=self.config.device if self.config.device is not None else "auto",\n        )\n',
        '            device=self.config.device if self.config.device is not None else "auto",\n'
        "            use_gated_attn=self.config.use_gated_attn,\n"
        "            gated_attn_headwise=self.config.gated_attn_headwise,\n"
        "            gated_attn_bias_init=self.config.gated_attn_bias_init,\n"
        "        )\n",
        "pass flags",
    )
    t = sub(
        t,
        "        self.prefix_length = self.config.prefix_length\n        self.rtc_processor = rtc_processor\n",
        "        self.prefix_length = self.config.prefix_length\n        self.rtc_processor = rtc_processor\n"
        "        self.float()  # fp32 everywhere (T4 has no native bf16)\n",
        "fp32 cast",
    )
    return t


PATCHES = {
    "configuration_smolvla.py": patch_config,
    "smolvlm_with_expert.py": patch_vlm,
    "modeling_smolvla.py": patch_modeling,
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", default=None)
    ap.add_argument("--revert", action="store_true")
    args = ap.parse_args()
    d = pathlib.Path(args.dir) if args.dir else find_dir()
    print("SmolVLA dir:", d)

    if args.revert:
        for name in PATCHES:
            bak = d / (name + ".orig")
            if bak.exists():
                shutil.copy(bak, d / name)
                print("restored", name)
        return

    if "use_gated_attn" in (d / "configuration_smolvla.py").read_text():
        print("Already patched, nothing to do.")
        return

    new = {}
    for name, fn in PATCHES.items():  # patch in memory first, write only if ALL succeed
        new[name] = fn((d / name).read_text())
    for name, text in new.items():
        shutil.copy(d / name, d / (name + ".orig"))
        (d / name).write_text(text)
        print("patched", name)


if __name__ == "__main__":
    main()

Writing /kaggle/working/gate_patch.py


In [4]:
%%writefile /kaggle/working/gate_sanity_test.py
#!/usr/bin/env python
"""
Safety tests for the gated-attention patch (run AFTER gate_patch.py, BEFORE training).

    python gate_sanity_test.py            # CPU unit tests of the gate module
    python gate_sanity_test.py --model    # + model-level tests on lerobot/smolvla_base (needs a GPU)

Model-level:
  (1) gate forced to 1.0 (W=0, b=50) reproduces the baseline exactly (train loss + sampled actions)
  (2) random gates: full uncached forward == prefill + cached denoise_step  (gate only touches expert tokens)
  (3) every gate (self- and cross-attn layers) gets a finite, non-zero gradient at init
"""
import argparse

import torch

from lerobot.policies.smolvla.smolvlm_with_expert import AttnOutputGate

MODEL_ID = "lerobot/smolvla_base"


def test_math():
    torch.manual_seed(0)
    B, L, E, H, D = 2, 7, 24, 3, 8
    x, y = torch.randn(B, L, E), torch.randn(B, L, H * D)
    for headwise in (False, True):
        g = AttnOutputGate(E, H, D, headwise=headwise, bias_init=0.0)
        torch.nn.init.normal_(g.proj.weight, std=0.5)
        out = g(x, y)
        s = torch.sigmoid(g.proj(x))
        if headwise:
            s = s.repeat_interleave(D, dim=-1)
        assert out.shape == y.shape and torch.allclose(out, y * s, atol=1e-6)
        # query dependence + causality of the gate: token t's output only depends on token t's x
        x2 = x.clone()
        x2[:, 4:] += 1.0
        assert torch.allclose(g(x, y)[:, :4], g(x2, y)[:, :4])
    g = AttnOutputGate(E, H, D, bias_init=4.0)
    assert (g(x, y) / y).mean() > 0.95, "default init should be near identity"
    print("[ok] math tests: elementwise/headwise formula, per-token dependence, near-identity init")


def fake_batch(policy, B=2, device="cuda"):
    from lerobot.utils.constants import ACTION, OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS, OBS_STATE

    cfg = policy.config
    batch = {}
    for key, ft in cfg.image_features.items():
        c, h, w = ft.shape
        batch[key] = torch.rand(B, c, h, w, device=device)
    batch[OBS_STATE] = torch.randn(B, cfg.robot_state_feature.shape[0], device=device)
    batch[ACTION] = torch.randn(B, cfg.chunk_size, cfg.action_feature.shape[0], device=device)
    batch[OBS_LANGUAGE_TOKENS] = torch.randint(0, 1000, (B, 16), device=device)
    batch[OBS_LANGUAGE_ATTENTION_MASK] = torch.ones(B, 16, dtype=torch.bool, device=device)
    return batch


def load_policy(use_gate, device="cuda"):
    from lerobot.configs import PreTrainedConfig
    from lerobot.policies.smolvla import SmolVLAPolicy

    cfg = PreTrainedConfig.from_pretrained(MODEL_ID)
    cfg.device = device
    cfg.use_gated_attn = use_gate
    return SmolVLAPolicy.from_pretrained(MODEL_ID, config=cfg).to(device)


def unpack(policy, batch):
    from lerobot.utils.constants import OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS

    images, img_masks = policy.prepare_images(batch)
    state = policy.prepare_state(batch)
    actions = policy.prepare_action(batch)
    return images, img_masks, batch[OBS_LANGUAGE_TOKENS], batch[OBS_LANGUAGE_ATTENTION_MASK], state, actions


@torch.no_grad()
def uncached_vt(vf, images, img_masks, lang, lm, state, x_t, time):
    from lerobot.policies.common.vla_utils import make_att_2d_masks

    pe, ppm, pam = vf.embed_prefix(images, img_masks, lang, lm, state=state)
    se, spm, sam = vf.embed_suffix(x_t, time)
    pad = torch.cat([ppm, spm], 1)
    att = torch.cat([pam, sam], 1)
    (_, suf), _ = vf.vlm_with_expert.forward(
        attention_mask=make_att_2d_masks(pad, att),
        position_ids=torch.cumsum(pad, 1) - 1,
        past_key_values=None,
        inputs_embeds=[pe, se],
        use_cache=False,
    )
    return vf.action_out_proj(suf[:, -vf.config.chunk_size :].float())


@torch.no_grad()
def cached_vt(vf, images, img_masks, lang, lm, state, x_t, time):
    from lerobot.policies.common.vla_utils import make_att_2d_masks

    pe, ppm, pam = vf.embed_prefix(images, img_masks, lang, lm, state=state)
    _, pkv = vf.vlm_with_expert.forward(
        attention_mask=make_att_2d_masks(ppm, pam),
        position_ids=torch.cumsum(ppm, 1) - 1,
        past_key_values=None,
        inputs_embeds=[pe, None],
        use_cache=True,
    )
    return vf.denoise_step(prefix_pad_masks=ppm, past_key_values=pkv, x_t=x_t, timestep=time)


def test_model():
    from lerobot.policies.common.flow_matching import FlowConvention, make_flow_matching_inputs

    dev = "cuda"
    base = load_policy(False, dev).eval()
    gated = load_policy(True, dev).eval()
    miss = [k for k in gated.state_dict() if k not in base.state_dict()]
    assert miss and all("attn_gates" in k for k in miss), f"unexpected extra keys: {miss[:5]}"
    gates = gated.model.vlm_with_expert.attn_gates
    print(f"[info] {len(miss)} new tensors, all attn_gates.*  ({len(gates)} gates, "
          f"{sum(p.numel() for p in gates.parameters()) / 1e6:.2f}M params)")
    assert next(gated.parameters()).dtype == torch.float32, "model is not fp32"

    torch.manual_seed(0)
    batch = fake_batch(base, device=dev)
    B = batch[next(iter(base.config.image_features))].shape[0]
    noise = torch.randn(B, base.config.chunk_size, base.config.max_action_dim, device=dev)
    time = torch.rand(B, device=dev) * 0.9 + 0.05
    ia, ib = unpack(base, dict(batch)), unpack(gated, dict(batch))

    # (1) gate == 1.0  ->  baseline, exactly
    for g in gates.values():
        g.proj.weight.data.zero_()
        g.proj.bias.data.fill_(50.0)
    with torch.no_grad():
        la = base.model.forward(*ia, noise, time)
        lb = gated.model.forward(*ib, noise, time)
        assert torch.allclose(la, lb, atol=1e-6), (la - lb).abs().max()
        sa = base.model.sample_actions(*ia[:5], noise=noise)
        sb = gated.model.sample_actions(*ib[:5], noise=noise)
        assert torch.allclose(sa, sb, atol=1e-6), (sa - sb).abs().max()
    print("[ok] (1) gate=1 reproduces the baseline (loss + sampled actions)")
    v_ref = None

    # (3) default init: every gate gets a finite non-zero gradient
    for g in gates.values():
        torch.nn.init.normal_(g.proj.weight, std=0.02)
        torch.nn.init.constant_(g.proj.bias, 4.0)
    gated.train()
    gated.zero_grad()
    loss = gated.model.forward(*ib, noise, time).mean()
    loss.backward()
    for k, g in gates.items():
        gr = g.proj.weight.grad
        assert gr is not None and torch.isfinite(gr).all() and gr.abs().sum() > 0, f"gate {k}: bad grad"
    assert torch.isfinite(loss)
    print(f"[ok] (3) all {len(gates)} gates (self+cross layers) get finite non-zero grads, loss={loss.item():.4f}")
    gated.eval()
    gated.zero_grad(set_to_none=True)

    # (2) random gates: uncached training path == prefill + cached denoise path
    for g in gates.values():
        torch.nn.init.normal_(g.proj.weight, std=0.5)
        torch.nn.init.normal_(g.proj.bias, std=1.0)
    x_t, _, _ = make_flow_matching_inputs(ib[5], noise, time, convention=FlowConvention.NOISE_AT_ONE)
    v_unc = uncached_vt(gated.model, *ib[:5], x_t, time)
    v_cac = cached_vt(gated.model, *ib[:5], x_t, time)
    err = (v_unc - v_cac).abs().max().item()
    assert err < 1e-3, f"train path and cached path disagree (max abs err {err:.3e})"
    for g in gates.values():
        g.proj.weight.data.zero_()
        g.proj.bias.data.fill_(50.0)
    v1 = uncached_vt(gated.model, *ib[:5], x_t, time)
    assert (v1 - v_unc).abs().max() > 1e-4, "gates have no effect"
    print(f"[ok] (2) uncached vs cached path agree (max abs err {err:.2e}); gates change the output")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", action="store_true")
    args = ap.parse_args()
    test_math()
    if args.model:
        test_model()
    print("ALL TESTS PASSED")

Writing /kaggle/working/gate_sanity_test.py


In [9]:
%%writefile /kaggle/working/gate_profile.py
#!/usr/bin/env python
"""
Parameter count + FLOPs for SmolVLA, baseline vs gated attention (run AFTER gate_patch.py).

    python gate_profile.py              # baseline vs gated, batch size 1
    python gate_profile.py --bs 8       # per-batch numbers for another batch size

FLOPs are measured with torch.utils.flop_counter.FlopCounterMode (counts matmul / bmm / conv / sdpa,
1 MAC = 2 FLOPs; elementwise ops like sigmoid/softmax are not counted).
  * train step = forward + backward of the flow-matching loss (what is actually executed, frozen VLM included)
  * inference  = embed prefix + VLM prefill once + `num_steps` (10) expert denoise steps = one action chunk
"""
import argparse
import gc
from collections import OrderedDict

import torch
from torch.utils.flop_counter import FlopCounterMode

from gate_sanity_test import fake_batch, load_policy, unpack


def count_params(policy):
    groups = OrderedDict(
        [("VLM vision encoder", 0), ("VLM connector", 0), ("VLM text model (16 layers)", 0),
         ("Action expert (lm_expert)", 0), ("Attention gates (new)", 0), ("Projections (state/action/time)", 0)]
    )
    train = OrderedDict((k, 0) for k in groups)
    for n, p in policy.named_parameters():
        if "attn_gates" in n:
            k = "Attention gates (new)"
        elif "lm_expert" in n:
            k = "Action expert (lm_expert)"
        elif "vision_model" in n:
            k = "VLM vision encoder"
        elif "connector" in n:
            k = "VLM connector"
        elif ".vlm." in n:
            k = "VLM text model (16 layers)"
        else:
            k = "Projections (state/action/time)"
        groups[k] += p.numel()
        if p.requires_grad:
            train[k] += p.numel()
    return groups, train


def count_flops(policy, bs, dev):
    batch = fake_batch(policy, B=bs, device=dev)
    inp = unpack(policy, batch)

    policy.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        policy.model.sample_actions(*inp[:5])
    infer = fc.get_total_flops()

    policy.train()
    policy.zero_grad(set_to_none=True)
    with FlopCounterMode(display=False) as fc:
        policy.model.forward(*inp).mean().backward()
    train = fc.get_total_flops()
    policy.zero_grad(set_to_none=True)

    policy.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        policy.model.forward(*inp)
    fwd = fc.get_total_flops()
    return {"forward (loss only)": fwd, "train step (fwd+bwd)": train, "inference (1 chunk, 10 steps)": infer}


def profile(use_gate, bs, dev):
    pol = load_policy(use_gate, dev)
    groups, trainable = count_params(pol)
    flops = count_flops(pol, bs, dev)
    del pol
    gc.collect()
    torch.cuda.empty_cache()
    return groups, trainable, flops


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bs", type=int, default=1)
    args = ap.parse_args()
    dev = "cuda"

    res = {name: profile(flag, args.bs, dev) for name, flag in [("baseline", False), ("gated", True)]}

    print(f"\n================ PARAMETERS ================")
    print(f"{'group':36s} {'baseline':>14s} {'gated':>14s}   (trainable gated)")
    gb, tb, _ = res["baseline"]
    gg, tg, _ = res["gated"]
    for k in gb:
        print(f"{k:36s} {gb[k]:>14,d} {gg[k]:>14,d}   {tg[k]:>14,d}")
    print(f"{'TOTAL':36s} {sum(gb.values()):>14,d} {sum(gg.values()):>14,d}   {sum(tg.values()):>14,d}")
    print(f"{'TRAINABLE':36s} {sum(tb.values()):>14,d} {sum(tg.values()):>14,d}")
    extra = sum(gg.values()) - sum(gb.values())
    print(f"-> gates add {extra:,d} params ({100 * extra / sum(gb.values()):.2f}% of the baseline total, "
          f"{100 * extra / sum(tb.values()):.2f}% of its trainable params)")

    print(f"\n================ FLOPs (batch size {args.bs}) ================")
    fb, fg = res["baseline"][2], res["gated"][2]
    print(f"{'':32s} {'baseline GFLOPs':>16s} {'gated GFLOPs':>14s} {'overhead':>10s}")
    for k in fb:
        print(f"{k:32s} {fb[k] / 1e9:>16,.2f} {fg[k] / 1e9:>14,.2f} {100 * (fg[k] / fb[k] - 1):>9.2f}%")
    print("(1 MAC = 2 FLOPs; matmul/bmm/conv/sdpa only)")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/gate_profile.py


In [10]:
!python {CODE_DIR}/gate_patch.py

SmolVLA dir: /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla
Already patched, nothing to do.


In [11]:
# safety tests: gate math (CPU) + model-level (gate=1 == baseline, cached == uncached path, grads on all gates)
!python {CODE_DIR}/gate_sanity_test.py --model

[ok] math tests: elementwise/headwise formula, per-token dependence, near-identity init
Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2565.68it/s]
Reducing the number of VLM layers to 16 ...
Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2583.68it/s]
Reducing the number of VLM layers to 16 ...
[info] 32 new tensors, all attn_gates.*  (16 gates, 11.07M params)
[ok] (1) gate=1 reproduces the baseline (loss + sampled actions)
[ok] (3) all 16 gates (self+cross layers) get finite non-zero grads, loss=0.9125
[ok] (2) uncached vs cached path agree (max abs err 1.91e-06); gates change the output
ALL TESTS PASSED


In [12]:
!python {CODE_DIR}/gate_profile.py --bs 1

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2602.30it/s]
Reducing the number of VLM layers to 16 ...
Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2554.87it/s]
Reducing the number of VLM layers to 16 ...

================ PARAMETERS ================
group                                      baseline          gated   (trainable gated)
VLM vision encoder                       86,433,024     86,433,024                0
VLM connector                            11,796,480     11,796,480                0
VLM text model (16 layers)              251,935,680    251,935,680                0
Action expert (lm_expert)                98,245,840     98,245,840       98,245,840
Attention gates (new)                             0     11,074,560       11,074,560
Proj

## Backbone swap (NEW)
`lfm_backbone_adapter.py` patches SmolVLA's `VLAFlowMatching` **at runtime** (no edits to lerobot files, on top of the gate patch above): it builds the frozen LFM2.5-VL-450M (8 layers) + the gated-KV conditioner, and feeds the conditioner's K/V to the *unchanged* gated action expert through `denoise_step`. `lfm_train_entry.py` installs it and then calls `lerobot-train`'s own `main()`, so data loading, normalisation, DDP, optimizer and scheduler stay exactly as before.

In [ ]:
%%writefile /kaggle/working/lfm_backbone_adapter.py
#!/usr/bin/env python
"""
Backbone swap for the gated-attention SmolVLA:  SmolVLM2  ->  frozen LFM2.5-VL-450M (first 8 LM layers).

Adapter pattern extracted from lfm2-backbone-new-dataset.ipynb ("F12 AdaptiveConditionalGatedKV"):

    3 camera images + task  --(frozen LFM2.5-VL, 8 layers)-->  hidden states [B, L, 1024]
        --(AdaptiveConditionalGatedKV, trainable)-->  per-layer K/V prefix cache for the action expert
        --(SmolVLA action expert with Gated Attention, unchanged)-->  velocity

Only SmolVLA's flow-matching model (VLAFlowMatching) is touched, at class level, at runtime:
  * __init__       : additionally builds `self.lfm_adapter` (frozen LFM backbone + trainable conditioner)
  * forward        : same flow-matching loss (x_t = t*noise + (1-t)*actions, target = noise - actions),
                     prefix KV comes from the adapter instead of the SmolVLM2 prefill
  * sample_actions : same 10-step Euler integration (t: 1 -> 0), prefix KV comes from the adapter
Untouched: action expert + gates, state/action/time projections, loss, data pipeline, optimizer, scheduler,
lerobot-train (it is simply started through lfm_train_entry.py after install()).

Note: the SmolVLM2 VLM stays inside the policy exactly as in the reference notebook (it provides the KV geometry
and the pretrained smolvla_base checkpoint layout) but it is inactive in the forward path.
"""
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

BACKBONE_ID = os.environ.get("LFM_BACKBONE_ID", "LiquidAI/LFM2.5-VL-450M")
KEEP_LAYERS = int(os.environ.get("LFM_KEEP_LAYERS", "8"))
VIS_TOKENS_PER_CAM = int(os.environ.get("LFM_VIS_TOKENS_PER_CAM", "48"))  # same budget as the LFM2 reference
IMG_SIZE = int(os.environ.get("LFM_IMG_SIZE", "256"))  # dataset frames are 256x256 (SmolVLA upsamples to 512 first)
# image order inside SmolVLA = camera1, camera2, camera3 = cam_head, cam_wrist, cam_side (see --rename_map)
CAM_LABELS = ("Head camera: ", "\nWrist camera: ", "\nSide camera: ")


# --------------------------------------------------------------------------- adapter (from the LFM2 notebook)
class AdaptiveConditionalGatedKV(nn.Module):
    """Identical to the reference notebook, except that `forward` also accepts a padding mask (needed because
    samples in a batch have different task lengths) and returns per-layer K/V lists instead of a cache object.
    With mask=None the maths is exactly the reference."""

    def __init__(self, input_dim, state_dim, kv_dim, kv_heads, head_dim, num_layers):
        super().__init__()
        self.input_dim, self.kv_dim = input_dim, kv_dim
        self.kv_heads, self.head_dim, self.num_layers = kv_heads, head_dim, num_layers
        self.mapper = nn.Sequential(nn.Linear(input_dim, input_dim), nn.SiLU(), nn.LayerNorm(input_dim))
        self.state_bridge = nn.Linear(state_dim, input_dim)
        self.kv_projection = nn.Linear(input_dim, 2 * kv_dim)
        self.gate_network = nn.Sequential(nn.Linear(input_dim, 128), nn.SiLU(), nn.Linear(128, num_layers))
        self.layer_bias = nn.Parameter(torch.zeros(num_layers, 2, kv_dim))

    def forward(self, vl_features, state_features, mask=None):
        x = self.mapper(vl_features.float())
        state_token = self.state_bridge(state_features.float()).unsqueeze(1)
        x = torch.cat([x, state_token], dim=1)
        B, P, _ = x.shape

        if mask is None:
            pooled = x.mean(dim=1)
        else:
            m = torch.cat([mask.to(x.dtype), torch.ones(B, 1, dtype=x.dtype, device=x.device)], dim=1).unsqueeze(-1)
            pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
        gates = torch.sigmoid(self.gate_network(pooled))

        base_k, base_v = self.kv_projection(x).chunk(2, dim=-1)
        ks, vs = [], []
        for layer_id in range(self.num_layers):
            gate = gates[:, layer_id].reshape(B, 1, 1)
            k = (base_k + self.layer_bias[layer_id, 0]) * gate
            v = (base_v + self.layer_bias[layer_id, 1]) * gate
            ks.append(k.reshape(B, P, self.kv_heads, self.head_dim).transpose(1, 2).contiguous())
            vs.append(v.reshape(B, P, self.kv_heads, self.head_dim).transpose(1, 2).contiguous())
        return ks, vs


class LFMAdapter(nn.Module):
    """Frozen LFM2.5-VL (first KEEP_LAYERS LM layers) + trainable AdaptiveConditionalGatedKV."""

    def __init__(self, model):
        super().__init__()
        from transformers import AutoProcessor, AutoTokenizer, Lfm2VlForConditionalGeneration

        ve = model.vlm_with_expert
        vcfg = ve.vlm.config.text_config  # original SmolVLA KV geometry (what the expert cross-attention expects)
        kv_heads, head_dim = int(vcfg.num_key_value_heads), int(vcfg.head_dim)
        self.num_layers = len(ve.lm_expert.layers)
        self.cache_format = "dynamic"
        self.bb_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

        backbone = Lfm2VlForConditionalGeneration.from_pretrained(
            BACKBONE_ID, dtype=self.bb_dtype, low_cpu_mem_usage=True
        )
        self.processor = AutoProcessor.from_pretrained(BACKBONE_ID)

        language = backbone.model.language_model
        n_full = len(language.layers)
        assert n_full >= KEEP_LAYERS, f"{BACKBONE_ID} has only {n_full} LM layers"
        layer_types = list(backbone.config.text_config.layer_types)
        language.layers = nn.ModuleList(list(language.layers[:KEEP_LAYERS]))  # FIRST pretrained layers
        for cfg in (language.config, backbone.config.text_config):
            cfg.num_hidden_layers = KEEP_LAYERS
            cfg.layer_types = layer_types[:KEEP_LAYERS]
        backbone.lm_head = nn.Identity()  # only hidden states are used
        backbone.requires_grad_(False)
        backbone.eval()
        self.backbone = backbone
        self.lfm_dim = int(backbone.config.text_config.hidden_size)

        self.conditioner = AdaptiveConditionalGatedKV(
            input_dim=self.lfm_dim,
            state_dim=model.state_proj.out_features,
            kv_dim=kv_heads * head_dim,
            kv_heads=kv_heads,
            head_dim=head_dim,
            num_layers=self.num_layers,
        ).float()

        # task text is recovered from SmolVLA's own tokenisation (SmolVLM2 tokenizer)
        tok = getattr(getattr(ve, "processor", None), "tokenizer", None)
        self.task_tokenizer = tok or AutoTokenizer.from_pretrained(model.config.vlm_model_name)
        self._text_ids = {}

        print(f"[LFMAdapter] {BACKBONE_ID}: {KEEP_LAYERS}/{n_full} LM layers kept, hidden={self.lfm_dim}, "
              f"layer_types={layer_types[:KEEP_LAYERS]}, backbone dtype={self.bb_dtype}", flush=True)

    # keep the frozen backbone in eval mode and out of checkpoints (it is re-downloaded from the hub on load)
    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def state_dict(self, *args, destination=None, prefix="", keep_vars=False):
        return self.conditioner.state_dict(destination=destination, prefix=prefix + "conditioner.", keep_vars=keep_vars)

    @property
    def device(self):
        return next(self.backbone.parameters()).device

    # ----------------------------------------------------------------- frozen feature extraction
    def _embed_text(self, text, language):
        if text not in self._text_ids:
            ids = self.processor.tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
            self._text_ids[text] = ids
        return language.embed_tokens(self._text_ids[text].to(self.device))

    def _to_pil(self, x):  # x: [3,H,W] in [-1,1]  (SmolVLA's prepare_images convention)
        x = ((x.detach().float() + 1.0) / 2.0).clamp(0, 1)
        if x.shape[-1] != IMG_SIZE or x.shape[-2] != IMG_SIZE:
            x = F.interpolate(x[None], size=(IMG_SIZE, IMG_SIZE), mode="bilinear", antialias=True, align_corners=False)[0]
        arr = (x.permute(1, 2, 0) * 255).round().byte().cpu().numpy()
        return Image.fromarray(arr)

    def _embed_image(self, x):
        inputs = self.processor.image_processor(images=self._to_pil(x), return_tensors="pt")
        required = ("pixel_values", "spatial_shapes", "pixel_attention_mask")
        assert all(k in inputs for k in required), f"LFM image preprocessing mismatch: {list(inputs.keys())}"
        prepared = {}
        for k in required:
            t = inputs[k].to(self.device)
            prepared[k] = t.to(self.bb_dtype) if k == "pixel_values" else t
        with torch.autocast("cuda", dtype=self.bb_dtype):
            out = self.backbone.get_image_features(**prepared, return_dict=True)
        visual = out.pooler_output
        if isinstance(visual, (list, tuple)):
            visual = torch.cat(list(visual), dim=0)
        if visual.ndim == 3:
            visual = visual.reshape(-1, visual.shape[-1])
        assert visual.ndim == 2 and visual.shape[-1] == self.lfm_dim
        visual = F.adaptive_avg_pool1d(visual.float().transpose(0, 1).unsqueeze(0), VIS_TOKENS_PER_CAM).transpose(1, 2)
        return visual.to(self.bb_dtype)  # [1, 48, D]

    def _task_text(self, tokens, mask):
        ids = tokens[mask.bool()].tolist()
        return self.task_tokenizer.decode(ids, skip_special_tokens=True).strip()

    @torch.no_grad()
    def features(self, images, img_masks, lang_tokens, lang_masks):
        """images: list of [B,3,H,W] in [-1,1]; returns (hidden [B,L,D] fp32, mask [B,L] bool)."""
        language = self.backbone.model.language_model
        B = images[0].shape[0]
        seqs = []
        for b in range(B):
            parts = []
            for c, img in enumerate(images):
                if not bool(img_masks[c][b]):
                    continue
                label = CAM_LABELS[c] if c < len(CAM_LABELS) else f"\nCamera {c + 1}: "
                parts.append(self._embed_text(label, language))
                parts.append(self._embed_image(img[b]))
            parts.append(self._embed_text("\n" + self._task_text(lang_tokens[b], lang_masks[b]), language))
            seqs.append(torch.cat([p.to(self.bb_dtype) for p in parts], dim=1)[0])

        L, D = max(s.shape[0] for s in seqs), seqs[0].shape[-1]
        x = torch.zeros(B, L, D, dtype=self.bb_dtype, device=self.device)
        m = torch.zeros(B, L, dtype=torch.bool, device=self.device)
        for b, s in enumerate(seqs):  # right padding
            x[b, : s.shape[0]] = s
            m[b, : s.shape[0]] = True
        with torch.autocast("cuda", dtype=self.bb_dtype):
            out = language(inputs_embeds=x, attention_mask=m.long(), use_cache=False, return_dict=True)
        hidden = out.last_hidden_state.float()
        assert torch.isfinite(hidden).all(), "non-finite LFM features"
        return hidden, m

    # ----------------------------------------------------------------- prefix KV cache for the action expert
    def make_cache(self, model, images, img_masks, lang_tokens, lang_masks, state):
        feats, mask = self.features(images, img_masks, lang_tokens, lang_masks)
        state_features = model.state_proj(state.float())
        ks, vs = self.conditioner(feats, state_features.to(feats.device), mask.to(feats.device))
        B = ks[0].shape[0]
        prefix_mask = torch.cat([mask, torch.ones(B, 1, dtype=torch.bool, device=mask.device)], dim=1)
        if self.cache_format == "dict":
            cache = {i: {"key_states": k, "value_states": v} for i, (k, v) in enumerate(zip(ks, vs))}
        else:
            from transformers import DynamicCache

            cache = DynamicCache()
            for i, (k, v) in enumerate(zip(ks, vs)):
                cache.update(k, v, i)
        return cache, prefix_mask


# --------------------------------------------------------------------------- class-level installation
_ORIG = {}


def _flow_cls():
    import lerobot.policies.smolvla.modeling_smolvla as mod

    cls = getattr(mod, "VLAFlowMatching", None)
    if cls is None:
        for name, obj in vars(mod).items():
            if isinstance(obj, type) and issubclass(obj, nn.Module) and "FlowMatching" in name:
                cls = obj
    assert cls is not None, "cannot find SmolVLA's flow-matching class"
    return cls


@torch.no_grad()
def _detect_cache_format(model):
    """Ask the real SmolVLA prefill which cache type the installed lerobot hands to denoise_step."""
    try:
        ve = model.vlm_with_expert
        dev = next(ve.vlm.parameters()).device
        dt = next(ve.vlm.parameters()).dtype
        hid = int(ve.vlm.config.text_config.hidden_size)
        pe = torch.zeros(1, 4, hid, device=dev, dtype=dt)
        _, pkv = ve.forward(
            attention_mask=torch.ones(1, 4, 4, dtype=torch.bool, device=dev),
            position_ids=torch.arange(4, device=dev)[None],
            past_key_values=None,
            inputs_embeds=[pe, None],
            use_cache=True,
        )
        return "dict" if isinstance(pkv, dict) else "dynamic"
    except Exception as e:  # pragma: no cover
        print("[LFMAdapter] cache-format probe failed, falling back to DynamicCache:", repr(e), flush=True)
        return "dynamic"


def _lfm_forward(self, images, img_masks, lang_tokens, lang_masks, state, actions, noise=None, time=None, **kw):
    """Same flow-matching loss as SmolVLA (noise at t=1); only the prefix source differs."""
    if noise is None:
        noise = torch.randn(actions.shape, device=actions.device, dtype=actions.dtype)
    if time is None:
        time = self.sample_time(actions.shape[0], actions.device)
    t = time[:, None, None]
    x_t = t * noise + (1 - t) * actions
    u_t = noise - actions
    cache, prefix_mask = self.lfm_adapter.make_cache(self, images, img_masks, lang_tokens, lang_masks, state)
    v_t = self.denoise_step(prefix_pad_masks=prefix_mask, past_key_values=cache, x_t=x_t, timestep=time)
    return F.mse_loss(u_t, v_t, reduction="none")


def _lfm_sample_actions(self, images, img_masks, lang_tokens, lang_masks, state, noise=None, **kw):
    bsize, device = state.shape[0], state.device
    if noise is None:
        noise = torch.randn(bsize, self.config.chunk_size, self.config.max_action_dim, device=device, dtype=torch.float32)
    cache, prefix_mask = self.lfm_adapter.make_cache(self, images, img_masks, lang_tokens, lang_masks, state)
    n = int(self.config.num_steps)
    x_t = noise
    for i in range(n):
        time = torch.full((bsize,), 1.0 - i / n, dtype=torch.float32, device=device)
        v_t = self.denoise_step(prefix_pad_masks=prefix_mask, past_key_values=cache, x_t=x_t, timestep=time)
        x_t = x_t - v_t / n
    return x_t


def install():
    cls = _flow_cls()
    if cls in _ORIG:
        return cls
    _ORIG[cls] = (cls.__init__, cls.forward, cls.sample_actions)
    orig_init = cls.__init__

    def __init__(self, *a, **kw):
        orig_init(self, *a, **kw)
        adapter = LFMAdapter(self)
        adapter.cache_format = _detect_cache_format(self)
        print("[LFMAdapter] prefix cache format:", adapter.cache_format, flush=True)
        self.lfm_adapter = adapter

    cls.__init__ = __init__
    cls.forward = _lfm_forward
    cls.sample_actions = _lfm_sample_actions
    return cls


def uninstall():
    cls = _flow_cls()
    if cls in _ORIG:
        cls.__init__, cls.forward, cls.sample_actions = _ORIG.pop(cls)


In [ ]:
%%writefile /kaggle/working/lfm_train_entry.py
#!/usr/bin/env python
"""Entry point: `lerobot-train` with the LFM2.5-VL backbone adapter installed (same CLI/arguments as lerobot-train)."""
if __name__ == "__main__":  # guard: dataloader workers use the 'spawn' start method and re-import this file
    import lfm_backbone_adapter as A

    A.install()
    import lerobot.scripts.lerobot_train as T

    (getattr(T, "main", None) or T.train)()


In [ ]:
%%writefile /kaggle/working/lfm_adapter_test.py
#!/usr/bin/env python
"""
Safety tests for the LFM2.5-VL backbone swap (run AFTER gate_patch.py, BEFORE training; needs a GPU).

  (1) backbone: exactly 8 LM layers, fully frozen, SmolVLM2 VLM frozen
  (2) the whole path trains: finite loss, finite non-zero grads on conditioner, expert, ALL 16 gates, projections
  (3) sample_actions: right shape, finite, deterministic for a fixed noise, depends on the noise
  (4) checkpoints hold the conditioner but not the frozen backbone
  (5) padding: same sample alone vs. batched with a longer task (info only; bf16/fp16 backbone => small drift)
"""
import torch

import lfm_backbone_adapter as A
from gate_sanity_test import fake_batch, load_policy, unpack


def grad_ok(params):
    gs = [p.grad for p in params if p.requires_grad and p.grad is not None]
    return bool(gs) and all(torch.isfinite(g).all() for g in gs) and sum(float(g.abs().sum()) for g in gs) > 0


def main():
    dev = "cuda"
    A.install()
    pol = load_policy(True, dev)
    m, ad = pol.model, pol.model.lfm_adapter
    ve = m.vlm_with_expert

    # (1)
    n_layers = len(ad.backbone.model.language_model.layers)
    assert n_layers == A.KEEP_LAYERS == 8, n_layers
    assert not any(p.requires_grad for p in ad.backbone.parameters()), "LFM backbone is not frozen"
    assert not any(p.requires_grad for p in ve.vlm.parameters()), "SmolVLM2 VLM is not frozen"
    assert all(p.requires_grad for p in ad.conditioner.parameters())
    assert len(ve.attn_gates) == 16 and all(p.requires_grad for p in ve.attn_gates.parameters())
    print(f"[ok] (1) backbone {A.BACKBONE_ID}: {n_layers} LM layers, frozen; hidden={ad.lfm_dim}; cache={ad.cache_format}")

    torch.manual_seed(0)
    batch = fake_batch(pol, device=dev)
    inp = unpack(pol, dict(batch))
    B = inp[5].shape[0]
    noise = torch.randn(B, pol.config.chunk_size, pol.config.max_action_dim, device=dev)
    time = torch.rand(B, device=dev) * 0.9 + 0.05

    # (2)
    pol.train()
    pol.zero_grad(set_to_none=True)
    assert not ad.backbone.training, "backbone left eval mode"
    losses = m.forward(*inp, noise, time)
    assert losses.shape == noise.shape and torch.isfinite(losses).all()
    losses.mean().backward()
    groups = {
        "conditioner": ad.conditioner.parameters(),
        "action expert": ve.lm_expert.parameters(),
        "state_proj": m.state_proj.parameters(),
        "action_in_proj": m.action_in_proj.parameters(),
        "action_out_proj": m.action_out_proj.parameters(),
    }
    for name, ps in groups.items():
        assert grad_ok(list(ps)), f"bad grad: {name}"
    for k, g in ve.attn_gates.items():
        assert grad_ok(list(g.parameters())), f"gate {k}: bad grad"
    assert all(p.grad is None for p in ad.backbone.parameters()), "grad leaked into frozen backbone"
    print(f"[ok] (2) finite non-zero grads: conditioner, expert, 16/16 gates, projections; loss={losses.mean().item():.4f}")
    pol.zero_grad(set_to_none=True)

    # (3)
    pol.eval()
    with torch.no_grad():
        a1 = m.sample_actions(*inp[:5], noise=noise)
        a2 = m.sample_actions(*inp[:5], noise=noise)
        a3 = m.sample_actions(*inp[:5], noise=torch.randn_like(noise))
    assert a1.shape == noise.shape and torch.isfinite(a1).all()
    assert torch.allclose(a1, a2, atol=1e-4), (a1 - a2).abs().max()
    assert (a1 - a3).abs().max() > 1e-4
    print(f"[ok] (3) sample_actions {tuple(a1.shape)} finite, deterministic, noise-dependent")

    # (4)
    keys = list(pol.state_dict().keys())
    assert any("lfm_adapter.conditioner" in k for k in keys)
    assert not any("lfm_adapter.backbone" in k for k in keys)
    print("[ok] (4) state_dict: conditioner saved, frozen LFM backbone excluded")

    # (5) info only
    with torch.no_grad():
        images, img_masks, lang, lm, state, _ = inp
        lang2, lm2 = lang.clone(), lm.clone()
        lm2[1, :] = True
        lm2[0, 8:] = False
        n0 = noise[:1]
        solo = m.sample_actions([i[:1] for i in images], [x[:1] for x in img_masks], lang2[:1], lm2[:1], state[:1], noise=n0)
        both = m.sample_actions(images, img_masks, lang2, lm2, state, noise=noise)[:1]
    print(f"[info] (5) padding drift (solo vs batched, max abs): {(solo - both).abs().max().item():.3e}")
    print("ALL LFM ADAPTER TESTS PASSED")


if __name__ == "__main__":
    main()


In [ ]:
# NEW — safety tests: 8 frozen layers, grads on conditioner/expert/all 16 gates, sampling, checkpoint contents
!python {CODE_DIR}/lfm_adapter_test.py

## Evaluation — model complexity (NEW)
Parameters (total / trainable, per component) and FLOPs / MACs (forward, train step, one inference chunk) for the original gated SmolVLA vs the LFM2.5-VL-450M (8 L) variant, measured with the same counter. Results are also written to `/kaggle/working/complexity_metrics.csv`.

In [ ]:
%%writefile /kaggle/working/lfm_profile.py
#!/usr/bin/env python
"""
Model complexity: parameters + FLOPs/MACs of the gated SmolVLA with its ORIGINAL backbone (SmolVLM2) vs the
LFM2.5-VL-450M (8 layers) backbone. Run AFTER gate_patch.py.

    python lfm_profile.py --bs 1

FLOPs: torch.utils.flop_counter.FlopCounterMode (matmul / bmm / conv / sdpa; 1 MAC = 2 FLOPs; elementwise ops,
softmax, sigmoid, tokenisation and image pre-processing are NOT counted).
  forward      = flow-matching loss forward (frozen backbone included)
  train step   = forward + backward
  inference    = backbone + conditioner + 10 expert denoise steps = one action chunk
"""
import argparse
import gc
from collections import OrderedDict

import pandas as pd
import torch
from torch.utils.flop_counter import FlopCounterMode

import lfm_backbone_adapter as A
from gate_sanity_test import fake_batch, load_policy, unpack

GROUPS = [
    "SmolVLM2 vision+text (original backbone)",
    "LFM2.5-VL vision encoder (SigLIP2)",
    "LFM2.5-VL projector",
    "LFM2.5-VL text model (8 layers + embeddings)",
    "Gated-KV conditioner (adapter)",
    "Action expert (lm_expert)",
    "Attention gates (G1)",
    "Projections (state/action/time)",
]


def group_of(n):
    if "attn_gates" in n:
        return GROUPS[6]
    if "lfm_adapter.backbone" in n:
        if "vision_tower" in n or "vision_model" in n:
            return GROUPS[1]
        if "projector" in n:
            return GROUPS[2]
        return GROUPS[3]
    if "lfm_adapter.conditioner" in n:
        return GROUPS[4]
    if "lm_expert" in n:
        return GROUPS[5]
    if ".vlm." in n:
        return GROUPS[0]
    return GROUPS[7]


def count_params(policy):
    total, train = OrderedDict((g, 0) for g in GROUPS), OrderedDict((g, 0) for g in GROUPS)
    for n, p in policy.named_parameters():
        g = group_of(n)
        total[g] += p.numel()
        if p.requires_grad:
            train[g] += p.numel()
    return total, train


def flops(fn):
    with FlopCounterMode(display=False) as fc:
        fn()
    return fc.get_total_flops()


def count_flops(policy, bs, dev, lfm):
    inp = unpack(policy, fake_batch(policy, B=bs, device=dev))
    out = {}
    policy.eval()
    with torch.no_grad():
        out["forward (loss only)"] = flops(lambda: policy.model.forward(*inp))
        out["inference (1 chunk, 10 steps)"] = flops(lambda: policy.model.sample_actions(*inp[:5]))
        if lfm:
            out["  of which LFM2.5-VL backbone (frozen)"] = flops(lambda: policy.model.lfm_adapter.features(*inp[:4]))
    policy.train()
    policy.zero_grad(set_to_none=True)
    out["train step (fwd+bwd)"] = flops(lambda: policy.model.forward(*inp).mean().backward())
    policy.zero_grad(set_to_none=True)
    return out


def profile(lfm, bs, dev):
    if lfm:
        A.install()
    else:
        A.uninstall()
    pol = load_policy(True, dev)
    total, train = count_params(pol)
    fl = count_flops(pol, bs, dev, lfm)
    del pol
    gc.collect()
    torch.cuda.empty_cache()
    return total, train, fl


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bs", type=int, default=1)
    ap.add_argument("--csv", default="/kaggle/working/complexity_metrics.csv")
    args = ap.parse_args()
    dev = "cuda"

    names = ["SmolVLM2 + gates (target, unchanged)", "LFM2.5-VL-450M 8L + gates (this notebook)"]
    res = [profile(False, args.bs, dev), profile(True, args.bs, dev)]
    rows = []

    print("\n================ PARAMETERS ================")
    print(f"{'group':46s} {names[0][:12]:>13s} {names[1][:12]:>13s}   trainable(lfm)")
    for g in GROUPS:
        a, b, t = res[0][0][g], res[1][0][g], res[1][1][g]
        print(f"{g:46s} {a:>13,d} {b:>13,d}   {t:>13,d}")
        rows.append({"metric": f"params: {g}", names[0]: a, names[1]: b, "trainable_lfm": t})
    tot = [sum(r[0].values()) for r in res]
    trn = [sum(r[1].values()) for r in res]
    inactive = res[1][0][GROUPS[0]]  # SmolVLM2 stays loaded but is not executed in the LFM variant
    act = [tot[0], tot[1] - inactive]
    for label, v in (("TOTAL (as instantiated)", tot), ("TRAINABLE", trn), ("ACTIVE in forward path", act)):
        print(f"{label:46s} {v[0]:>13,d} {v[1]:>13,d}")
        rows.append({"metric": f"params: {label}", names[0]: v[0], names[1]: v[1]})
    print("(ACTIVE = total minus the SmolVLM2 weights that are loaded in the LFM variant but never executed)")

    print(f"\n================ FLOPs / MACs (batch size {args.bs}) ================")
    fa, fb = res[0][2], res[1][2]
    print(f"{'':44s} {'GFLOPs base':>12s} {'GFLOPs lfm':>12s} {'GMACs base':>11s} {'GMACs lfm':>10s}")
    for k in fb:
        a = fa.get(k)
        b = fb[k]
        sa = f"{a / 1e9:>12,.2f}" if a is not None else f"{'-':>12s}"
        ma = f"{a / 2e9:>11,.2f}" if a is not None else f"{'-':>11s}"
        print(f"{k:44s} {sa} {b / 1e9:>12,.2f} {ma} {b / 2e9:>10,.2f}")
        rows.append({"metric": f"flops: {k.strip()}", names[0]: a, names[1]: b,
                     "GMACs_base": None if a is None else a / 2e9, "GMACs_lfm": b / 2e9})
    print("(1 MAC = 2 FLOPs; matmul/bmm/conv/sdpa only)")
    pd.DataFrame(rows).to_csv(args.csv, index=False)
    print("saved", args.csv)


if __name__ == "__main__":
    main()


In [ ]:
!python {CODE_DIR}/lfm_profile.py --bs 1

## Training (same hyper-parameters as the original notebook; only the entry script and output names differ)

In [ ]:
import os, shutil, subprocess, sys

ENTRY  = "/kaggle/working/lfm_train_entry.py"      # = lerobot-train + LFM2.5-VL backbone adapter
STEPS  = 20000
OUT    = "/kaggle/working/outputs/smolvla_gated_attn_lfm25_8L"
RENAME = ('{"observation.images.cam_head":"observation.images.camera1",'
          '"observation.images.cam_wrist":"observation.images.camera2",'
          '"observation.images.cam_side":"observation.images.camera3"}')

cmd = [
    "accelerate", "launch", "--multi_gpu", "--num_processes=2", "--mixed_precision=no", ENTRY,
    "--policy.path=lerobot/smolvla_base",
    "--policy.use_gated_attn=true",
    "--policy.gated_attn_headwise=false",   # false = elementwise gate (best in the paper); true = one scalar per head
    "--policy.gated_attn_bias_init=4.0",    # sigmoid(4)=0.98 -> starts ~identity from the pretrained expert
    f"--policy.scheduler_decay_steps={STEPS}",
    "--policy.push_to_hub=false",
    f"--dataset.repo_id={DATASET}",
    "--dataset.revision=main",
    "--dataset.video_backend=pyav",
    f"--rename_map={RENAME}",
    "--batch_size=8",
    "--num_workers=4",
    f"--steps={STEPS}",
    "--save_freq=2500",
    "--log_freq=50",
    f"--output_dir={OUT}",
    "--job_name=smolvla_gated_attn_lfm25_8L",
    "--seed=1000",
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in proc.stdout:
    print(line, end="")
sys.exit(proc.wait())